# Loading and Accessing Data

This notebook demonstrates how to load datasets from manifests or archives and access different types of protein data. We'll explore the Pythonic API for working with sequences, structures, assays, and MSAs.

## Loading Datasets

There are two main ways to load a PG2 dataset:
1. From a manifest file (TOML)
2. From a dataset archive (ZIP)

Let's start by importing the necessary modules:

In [1]:
from pathlib import Path
from pg2_dataset import Dataset, Manifest

# Set up paths
manifest_path = Path("../example_data/neime_2019.toml")

### Method 1: Loading from Manifest

In [2]:
# Load manifest first
try:
    manifest = Manifest.from_path(manifest_path)
    print(f"Loaded manifest: {manifest.name}")
    print(f"Version: {manifest.version}")
    
    # Create dataset from manifest
    dataset = Dataset.from_manifest(manifest)
    print(f"\nDataset created successfully!")
    print(f"Dataset name: {dataset.name}")
    
except Exception as e:
    print(f"Error loading from manifest: {e}")
    print("This might be due to missing data files or incorrect paths.")

Loaded manifest: NEIME_2019
Version: 1.0.0

Dataset created successfully!
Dataset name: NEIME_2019


### Method 2: Loading from Archive

If you have a dataset archive (created in the previous notebook), you can load it directly:

In [6]:
# Look for existing archives
archive_path = "../example_data/NEIME_2019.zip"

try:
    dataset = Dataset.from_path(archive_path)
except Exception as e:
    print(f"Error loading from archive: {e}")
    print(f"Did you create an archive in the previous tutorial?")

## Exploring Dataset Structure

Let's examine what's in our dataset:

In [4]:
if 'dataset' in locals():
    print(f"Dataset: {dataset.name}")
    print(f"Description: {dataset.description}")
    print("\nDataset contents:")
    print(f"  - Sequences: {len(dataset.sequences)}")
    print(f"  - Structures: {len(dataset.structures)}")
    print(f"  - MSAs: {len(dataset.msas)}")
    print(f"  - Assays: {len(dataset.assays)}")
    print(f"  - Assay conditions: {len(dataset.assay_conditions)}")
else:
    print("Dataset not loaded. Please check the previous cells.")

Dataset: NEIME_2019
Description: NEIME enzyme dataset from Kennouche et al. 2019 with DMS scores

Dataset contents:
  - Sequences: 1
  - Structures: 1
  - MSAs: 1
  - Assays: 1
  - Assay conditions: 2


## Accessing Assays

In [72]:
# Access the assays
assays = dataset.assays

# Extract an specific assay
my_assay = assays[0]

# We can get a summary of the data encoded in this assay:
for field in my_assay.__class__.model_fields:
    print(f"Found a field for {field}")
    print(f"Pythonic description of the field information:")
    print(f"{my_assay.__class__.model_fields[field]}")
    print(f"------------")

Found a field for name
Pythonic description of the field information:
annotation=str required=True description='The name of the assay.'
------------
Found a field for records
Pythonic description of the field information:
annotation=list[tuple[str, Union[int, float, bool, str]]] required=True description='The records of the assay, pairs of Sequence and target values.'
------------
Found a field for conditions
Pythonic description of the field information:
annotation=dict[str, Union[int, float, bool, str]] required=False default_factory=dict description='The conditions of the assay, defined in the manifest.'
------------
Found a field for description
Pythonic description of the field information:
annotation=Union[str, NoneType] required=False default=None description='The description of the assay.'
------------
Found a field for sequence_feature_name
Pythonic description of the field information:
annotation=str required=False default='sequence' description='The sequence feature name in 

In [69]:
my_assay.__class__.model_fields['name']

FieldInfo(annotation=str, required=True, description='The name of the assay.')

In [55]:
# Access specific attributes such as name
name = my_assay.name
print(f"Assay name: {name}")

# Or extract the records
records = my_assay.records
print(f"{name} contains {len(records)} records")
print(f"record 1: {records[0][0][:20]}... with value {records[0][1]}")

Assay name: Assay1
Assay1 contains 922 records
record 1: ITLIELMIVIAIVGILAAVA... with value -3.5980000000000003


## Accessing Assay Conditions

Assay conditions describe the experimental setup:

In [62]:
print(f"Number of assay conditions: {len(dataset.assay_conditions)}")
    
for i, condition in enumerate(dataset.assay_conditions):
    print(f"\nCondition {i+1}:")
    print(f"  - Name: {condition.name}")
    print(f"  - Description: {condition.description}")
    print(f"  - Unit: {condition.unit}")
    print(f"  - Value: {condition.value}")

Number of assay conditions: 2

Condition 1:
  - Name: temperature
  - Description: Reaction temperature
  - Unit: °C
  - Value: 37

Condition 2:
  - Name: pH
  - Description: Buffer pH
  - Unit: pH
  - Value: 7.4


## Accessing Structures

Structure data provides 3D information about proteins:

In [ ]:
if 'dataset' in locals() and dataset.structures:
    print(f"Number of structures: {len(dataset.structures)}")
    
    for i, structure in enumerate(dataset.structures):
        print(f"\nStructure {i+1}:")
        print(f"  - Name: {getattr(structure, 'name', 'N/A')}")
        print(f"  - Type: {type(structure)}")
        print(f"  - Value type: {type(structure.value)}")
        
        # Try to get structure information
        if hasattr(structure.value, 'get_structure'):
            print(f"  - BioPython structure object")
        elif hasattr(structure.value, '__len__'):
            print(f"  - Data length: {len(structure.value)}")
        
        # Check for metadata
        if hasattr(structure, 'metadata') and structure.metadata:
            print(f"  - Metadata: {structure.metadata}")
else:
    print("No structures found in dataset")

## Accessing MSAs (Multiple Sequence Alignments)

MSAs provide evolutionary information through aligned sequences:

In [ ]:
if 'dataset' in locals() and dataset.msas:
    print(f"Number of MSAs: {len(dataset.msas)}")
    
    for i, msa in enumerate(dataset.msas):
        print(f"\nMSA {i+1}:")
        print(f"  - Name: {getattr(msa, 'name', 'N/A')}")
        print(f"  - Type: {type(msa)}")
        print(f"  - Value type: {type(msa.value)}")
        
        # Try to get MSA information
        try:
            if hasattr(msa.value, '__len__'):
                print(f"  - Number of sequences: {len(msa.value)}")
            
            # If it's a BioPython MultipleSeqAlignment
            if hasattr(msa.value, 'get_alignment_length'):
                print(f"  - Alignment length: {msa.value.get_alignment_length()}")
            elif hasattr(msa.value, '__getitem__') and len(msa.value) > 0:
                first_seq = msa.value[0]
                if hasattr(first_seq, '__len__'):
                    print(f"  - Alignment length: {len(first_seq)}")
        except Exception as e:
            print(f"  - Could not determine MSA properties: {e}")
        
        # Check for metadata
        if hasattr(msa, 'metadata') and msa.metadata:
            print(f"  - Metadata: {msa.metadata}")
else:
    print("No MSAs found in dataset")

## Accessing Sequences

Sequences are the foundation of protein datasets. Let's explore the sequence data:

In [6]:
if 'dataset' in locals() and dataset.sequences:
    print(f"Number of sequences: {len(dataset.sequences)}")
    
    # Examine the first sequence
    seq = dataset.sequences[0]
    print(f"\nFirst sequence:")
    print(f"  - Type: {type(seq)}")
    print(f"  - Value type: {type(seq.value)}")
    
    # Access sequence data
    if hasattr(seq.value, '__len__'):
        print(f"  - Length: {len(seq.value)}")
    
    # Display first 50 characters of sequence
    if hasattr(seq.value, '__getitem__'):
        print(f"  - First 50 chars: {str(seq.value)[:50]}...")
    
    # Check for metadata
    if hasattr(seq, 'metadata') and seq.metadata:
        print(f"  - Metadata: {seq.metadata}")
else:
    print("No sequences found in dataset")

Number of sequences: 1

First sequence:
  - Type: <class 'pg2_dataset.models.sequence.Sequence'>
  - Value type: <class 'Bio.Seq.Seq'>
  - Length: 161
  - First 50 chars: FTLIELMIVIAIVGILAAVALPAYQDYTARAQVSEAILLAEGQKSAVTEY...


## Data Access Patterns for ML

Here are common patterns for accessing data in ML workflows:

### Pattern 1: Extract All Sequences and Targets

In [ ]:
def extract_ml_data(dataset):
    """Extract sequences and targets for ML training."""
    all_sequences = []
    all_targets = []
    
    for assay in dataset.assays:
        if hasattr(assay, 'records'):
            for record in assay.records:
                if hasattr(record, 'sequence') and hasattr(record, 'value'):
                    all_sequences.append(str(record.sequence))
                    all_targets.append(record.value)
    
    return all_sequences, all_targets

if 'dataset' in locals():
    sequences, targets = extract_ml_data(dataset)
    print(f"Extracted {len(sequences)} sequence-target pairs")
    
    if sequences:
        print(f"Sequence lengths: min={min(len(s) for s in sequences)}, max={max(len(s) for s in sequences)}")
        print(f"Target range: min={min(targets)}, max={max(targets)}")

### Pattern 2: Access Reference Sequence

In [ ]:
def get_reference_sequence(dataset):
    """Get the wild-type or reference sequence."""
    for sequence in dataset.sequences:
        # Check if this is a wild-type sequence
        if hasattr(sequence, 'sequence_type') and 'wild' in str(sequence.sequence_type).lower():
            return str(sequence.value)
    
    # If no wild-type found, return first sequence
    if dataset.sequences:
        return str(dataset.sequences[0].value)
    
    return None

if 'dataset' in locals():
    ref_seq = get_reference_sequence(dataset)
    if ref_seq:
        print(f"Reference sequence (length {len(ref_seq)}):")
        print(f"{ref_seq[:50]}...")
    else:
        print("No reference sequence found")

### Pattern 3: Access Structural Information

In [ ]:
def get_structure_info(dataset):
    """Get information about available structures."""
    structure_info = []
    
    for i, structure in enumerate(dataset.structures):
        info = {
            'index': i,
            'name': getattr(structure, 'name', f'Structure_{i}'),
            'type': type(structure.value).__name__,
        }
        
        # Try to get more specific information
        if hasattr(structure.value, 'get_structure'):
            info['format'] = 'BioPython'
        elif hasattr(structure.value, '__len__'):
            info['size'] = len(structure.value)
        
        structure_info.append(info)
    
    return structure_info

if 'dataset' in locals():
    struct_info = get_structure_info(dataset)
    if struct_info:
        print("Available structures:")
        for info in struct_info:
            print(f"  - {info['name']}: {info['type']}")
    else:
        print("No structures available")

## Advanced Data Access

### Filtering and Subsetting

In [ ]:
def filter_assay_data(dataset, min_value=None, max_value=None):
    """Filter assay data by value range."""
    filtered_data = []
    
    for assay in dataset.assays:
        if hasattr(assay, 'records'):
            for record in assay.records:
                if hasattr(record, 'value'):
                    value = record.value
                    if isinstance(value, (int, float)):
                        if (min_value is None or value >= min_value) and \
                           (max_value is None or value <= max_value):
                            filtered_data.append({
                                'sequence': str(record.sequence) if hasattr(record, 'sequence') else None,
                                'value': value
                            })
    
    return filtered_data

if 'dataset' in locals():
    # Filter for high-value variants (assuming higher is better)
    high_value_data = filter_assay_data(dataset, min_value=0)
    print(f"Found {len(high_value_data)} records with positive values")
    
    if high_value_data:
        values = [d['value'] for d in high_value_data]
        print(f"Value statistics: mean={sum(values)/len(values):.2f}, max={max(values):.2f}")

### Working with Metadata

In [ ]:
def explore_metadata(dataset):
    """Explore metadata across all data types."""
    metadata_summary = {
        'dataset': {},
        'sequences': [],
        'structures': [],
        'msas': [],
        'assays': []
    }
    
    # Dataset metadata
    if hasattr(dataset, 'metadata') and dataset.metadata:
        metadata_summary['dataset'] = dataset.metadata
    
    # Collect metadata from each data type
    for data_type, objects in [
        ('sequences', dataset.sequences),
        ('structures', dataset.structures),
        ('msas', dataset.msas),
        ('assays', dataset.assays)
    ]:
        for obj in objects:
            if hasattr(obj, 'metadata') and obj.metadata:
                metadata_summary[data_type].append(obj.metadata)
    
    return metadata_summary

if 'dataset' in locals():
    metadata = explore_metadata(dataset)
    print("Metadata summary:")
    for data_type, meta_list in metadata.items():
        if isinstance(meta_list, list):
            print(f"  {data_type}: {len(meta_list)} objects with metadata")
        else:
            print(f"  {data_type}: {len(meta_list)} metadata fields")

## Integration with ML Libraries

Here's how you might integrate PG2 Dataset with common ML libraries:

### PyTorch Dataset

In [ ]:
# Example PyTorch dataset (requires torch)
class PG2Dataset:
    """PyTorch-compatible dataset wrapper for PG2 data."""
    
    def __init__(self, pg2_dataset):
        self.sequences, self.targets = extract_ml_data(pg2_dataset)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

if 'dataset' in locals():
    pytorch_dataset = PG2Dataset(dataset)
    print(f"Created PyTorch dataset with {len(pytorch_dataset)} samples")
    
    if len(pytorch_dataset) > 0:
        sample_seq, sample_target = pytorch_dataset[0]
        print(f"Sample: {sample_seq[:30]}... -> {sample_target}")

### Scikit-learn Compatible Format

In [ ]:
def to_sklearn_format(dataset, feature_extractor=None):
    """Convert PG2 dataset to scikit-learn format."""
    sequences, targets = extract_ml_data(dataset)
    
    if feature_extractor is None:
        # Simple feature extraction: sequence length and amino acid counts
        def simple_features(seq):
            return [
                len(seq),
                seq.count('A'), seq.count('C'), seq.count('D'), seq.count('E'),
                seq.count('F'), seq.count('G'), seq.count('H'), seq.count('I'),
                seq.count('K'), seq.count('L'), seq.count('M'), seq.count('N'),
                seq.count('P'), seq.count('Q'), seq.count('R'), seq.count('S'),
                seq.count('T'), seq.count('V'), seq.count('W'), seq.count('Y')
            ]
        feature_extractor = simple_features
    
    X = [feature_extractor(seq) for seq in sequences]
    y = targets
    
    return X, y

if 'dataset' in locals():
    X, y = to_sklearn_format(dataset)
    print(f"Created sklearn format: X shape = ({len(X)}, {len(X[0]) if X else 0}), y length = {len(y)}")
    
    if X:
        print(f"Sample features: {X[0][:5]}... (first 5 features)")
        print(f"Sample target: {y[0]}")

## Best Practices for Data Access

### 1. Error Handling
Always check if data exists before accessing it:

```python
if dataset.assays:
    # Process assays
    pass
else:
    print("No assays available")
```

### 2. Memory Efficiency
For large datasets, consider processing data in chunks:

```python
def process_assay_chunks(assay, chunk_size=1000):
    for i in range(0, len(assay.records), chunk_size):
        chunk = assay.records[i:i+chunk_size]
        # Process chunk
        yield chunk
```

### 3. Data Validation
Validate data before using it in ML pipelines:

```python
def validate_sequences(sequences):
    valid_aa = set('ACDEFGHIKLMNPQRSTVWY')
    for seq in sequences:
        if not all(aa in valid_aa for aa in seq):
            print(f"Invalid sequence: {seq}")
```

### 4. Caching
Cache expensive operations:

```python
from functools import lru_cache

@lru_cache(maxsize=None)
def extract_features(sequence):
    # Expensive feature extraction
    return compute_features(sequence)
```

## Summary

In this notebook, we've learned how to:

1. **Load datasets** from manifests and archives
2. **Access different data types**: sequences, structures, MSAs, and assays
3. **Extract data for ML**: sequences, targets, and features
4. **Work with metadata** and conditions
5. **Integrate with ML libraries** like PyTorch and scikit-learn
6. **Apply best practices** for robust data access

The PG2 Dataset package provides a powerful and flexible way to work with protein data in machine learning workflows. The standardized API makes it easy to switch between different datasets while maintaining consistent code structure.

## Next Steps

Now you're ready to:
- Use PG2 Dataset in your own ML projects
- Create custom feature extractors for your specific needs
- Build reproducible protein engineering pipelines
- Share standardized datasets with collaborators

Happy protein engineering! 🧬